In [0]:
# =====================================
# PROJETO 01 - ENGENHARIA DE DADOS
# Pipeline de Vendas
# =====================================

print("Iniciando projeto de Engenharia de Dados!")

print("Databricks Free Edition funcionando!")

print("Ambiente preparado para Python e PySpark.")

Iniciando projeto de Engenharia de Dados!
Databricks Free Edition funcionando!
Ambiente preparado para Python e PySpark.


In [0]:
# =====================================
# ETAPA 2 - CRIANDO DADOS DE CLIENTES
# =====================================

clientes = [
    (1, "Carlos", "Porto Alegre"),
    (2, "Mariana", "São Paulo"),
    (3, "João", "Curitiba"),
    (4, "Fernanda", "Florianópolis"),
    (5, "Roberto", "Rio de Janeiro")
]

colunas = ["id_cliente", "nome", "cidade"]

df_clientes = spark.createDataFrame(clientes, colunas)

display(df_clientes)

id_cliente,nome,cidade
1,Carlos,Porto Alegre
2,Mariana,São Paulo
3,João,Curitiba
4,Fernanda,Florianópolis
5,Roberto,Rio de Janeiro


In [0]:
# =====================================
# ETAPA 3 - CRIANDO DADOS DE PRODUTOS
# =====================================

produtos = [
    (101, "Notebook", 3500.00),
    (102, "Smartphone", 2200.00),
    (103, "Televisao", 2800.00),
    (104, "Tablet", 1500.00),
    (105, "Monitor", 950.00),
    (106, "Teclado", 150.00),
    (107, "Mouse", 80.00),
    (108, "Impressora", 750.00)
]

colunas_produtos = ["id_produto", "produto", "preco"]

df_produtos = spark.createDataFrame(
    produtos,
    colunas_produtos
)

display(df_produtos)

id_produto,produto,preco
101,Notebook,3500.0
102,Smartphone,2200.0
103,Televisao,2800.0
104,Tablet,1500.0
105,Monitor,950.0
106,Teclado,150.0
107,Mouse,80.0
108,Impressora,750.0


In [0]:
# =====================================
# ETAPA 4 - CADASTRO DE VENDEDORES
# =====================================

vendedores = [
    (1, "Ana", "Sul"),
    (2, "Pedro", "Sudeste"),
    (3, "Lucas", "Centro-Oeste"),
    (4, "Juliana", "Nordeste"),
    (5, "Marcos", "Norte")
]

colunas_vendedores = [
    "id_vendedor",
    "nome_vendedor",
    "regiao"
]

df_vendedores = spark.createDataFrame(
    vendedores,
    colunas_vendedores
)

display(df_vendedores)

id_vendedor,nome_vendedor,regiao
1,Ana,Sul
2,Pedro,Sudeste
3,Lucas,Centro-Oeste
4,Juliana,Nordeste
5,Marcos,Norte


In [0]:
# =====================================
# ETAPA 5 - REGISTROS DE VENDAS
# =====================================

vendas = [
    (1001, 1, 101, 1, 3500.00),
    (1002, 2, 102, 2, 2200.00),
    (1003, 3, 103, 3, 2800.00),
    (1004, 4, 104, 4, 1500.00),
    (1005, 5, 105, 5, 950.00),
    (1006, 1, 106, 1, 150.00),
    (1007, 2, 107, 2, 80.00),
    (1008, 3, 108, 3, 750.00),
    (1009, 4, 101, 4, 3500.00),
    (1010, 5, 102, 5, 2200.00)
]

colunas_vendas = [
    "id_venda",
    "id_cliente",
    "id_produto",
    "id_vendedor",
    "valor_venda"
]

df_vendas = spark.createDataFrame(
    vendas,
    colunas_vendas
)

display(df_vendas)

id_venda,id_cliente,id_produto,id_vendedor,valor_venda
1001,1,101,1,3500.0
1002,2,102,2,2200.0
1003,3,103,3,2800.0
1004,4,104,4,1500.0
1005,5,105,5,950.0
1006,1,106,1,150.0
1007,2,107,2,80.0
1008,3,108,3,750.0
1009,4,101,4,3500.0
1010,5,102,5,2200.0


In [0]:
# =====================================
# ETAPA 6 - INTEGRACAO DAS TABELAS
# =====================================

from pyspark.sql.functions import col

df_vendas_completo = (
    df_vendas.alias("v")
    .join(
        df_clientes.alias("c"),
        col("v.id_cliente") == col("c.id_cliente")
    )
    .join(
        df_produtos.alias("p"),
        col("v.id_produto") == col("p.id_produto")
    )
    .join(
        df_vendedores.alias("vd"),
        col("v.id_vendedor") == col("vd.id_vendedor")
    )
    .select(
        col("v.id_venda"),
        col("c.nome").alias("cliente"),
        col("c.cidade"),
        col("p.produto"),
        col("p.preco"),
        col("vd.nome_vendedor"),
        col("vd.regiao"),
        col("v.valor_venda")
    )
)

display(df_vendas_completo)

id_venda,cliente,cidade,produto,preco,nome_vendedor,regiao,valor_venda
1001,Carlos,Porto Alegre,Notebook,3500.0,Ana,Sul,3500.0
1002,Mariana,São Paulo,Smartphone,2200.0,Pedro,Sudeste,2200.0
1003,João,Curitiba,Televisao,2800.0,Lucas,Centro-Oeste,2800.0
1004,Fernanda,Florianópolis,Tablet,1500.0,Juliana,Nordeste,1500.0
1005,Roberto,Rio de Janeiro,Monitor,950.0,Marcos,Norte,950.0
1006,Carlos,Porto Alegre,Teclado,150.0,Ana,Sul,150.0
1007,Mariana,São Paulo,Mouse,80.0,Pedro,Sudeste,80.0
1008,João,Curitiba,Impressora,750.0,Lucas,Centro-Oeste,750.0
1009,Fernanda,Florianópolis,Notebook,3500.0,Juliana,Nordeste,3500.0
1010,Roberto,Rio de Janeiro,Smartphone,2200.0,Marcos,Norte,2200.0


In [0]:
# =====================================
# ETAPA 7 - INDICADORES DE VENDAS
# =====================================

from pyspark.sql.functions import sum, count, avg, desc

# 1. Faturamento total
faturamento = df_vendas_completo.agg(
    sum("valor_venda").alias("faturamento_total")
)

display(faturamento)

# 2. Quantidade de vendas
quantidade = df_vendas_completo.agg(
    count("id_venda").alias("total_vendas")
)

display(quantidade)

# 3. Faturamento por vendedor
vendas_vendedor = (
    df_vendas_completo
    .groupBy("nome_vendedor")
    .agg(
        sum("valor_venda").alias("faturamento")
    )
    .orderBy(desc("faturamento"))
)

display(vendas_vendedor)

# 4. Faturamento por região
vendas_regiao = (
    df_vendas_completo
    .groupBy("regiao")
    .agg(
        sum("valor_venda").alias("faturamento")
    )
    .orderBy(desc("faturamento"))
)

display(vendas_regiao)

faturamento_total
17630.0


total_vendas
10


nome_vendedor,faturamento
Juliana,5000.0
Ana,3650.0
Lucas,3550.0
Marcos,3150.0
Pedro,2280.0


regiao,faturamento
Nordeste,5000.0
Sul,3650.0
Centro-Oeste,3550.0
Norte,3150.0
Sudeste,2280.0


In [0]:
# =====================================
# ETAPA 8 - CAMADA BRONZE
# =====================================

# Criando banco de dados do projeto
spark.sql("""
CREATE DATABASE IF NOT EXISTS projeto_vendas
""")

# Salvando os dados originais em Delta Lake

df_clientes.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.bronze_clientes"
)

df_produtos.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.bronze_produtos"
)

df_vendedores.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.bronze_vendedores"
)

df_vendas.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.bronze_vendas"
)

print("Camada Bronze criada com sucesso!")

Camada Bronze criada com sucesso!


In [0]:
# =====================================
# ETAPA 9 - CAMADA SILVER
# LIMPEZA DOS CLIENTES
# =====================================

from pyspark.sql.functions import col, trim, initcap

# Ler dados da camada Bronze
df_clientes_bronze = spark.table(
    "projeto_vendas.bronze_clientes"
)

# Limpar e padronizar
df_clientes_silver = (
    df_clientes_bronze
    .dropDuplicates(["id_cliente"])
    .filter(
        col("id_cliente").isNotNull() &
        col("nome").isNotNull()
    )
    .withColumn("nome", initcap(trim(col("nome"))))
    .withColumn("cidade", initcap(trim(col("cidade"))))
)

# Salvar na camada Silver
df_clientes_silver.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.silver_clientes"
)

display(df_clientes_silver)

id_cliente,nome,cidade
1,Carlos,Porto Alegre
2,Mariana,São Paulo
3,João,Curitiba
4,Fernanda,Florianópolis
5,Roberto,Rio De Janeiro


In [0]:
df_produtos_bronze = spark.table(
    "projeto_vendas.bronze_produtos"
)

df_produtos_silver = (
    df_produtos_bronze
    .dropDuplicates(["id_produto"])
    .filter(
        col("id_produto").isNotNull() &
        col("produto").isNotNull() &
        (col("preco") > 0)
    )
    .withColumn("produto", initcap(trim(col("produto"))))
)

df_produtos_silver.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.silver_produtos"
)

display(df_produtos_silver)

id_produto,produto,preco
101,Notebook,3500.0
102,Smartphone,2200.0
103,Televisao,2800.0
104,Tablet,1500.0
105,Monitor,950.0
106,Teclado,150.0
107,Mouse,80.0
108,Impressora,750.0


In [0]:
spark.sql("""
SHOW TABLES IN projeto_vendas
""").show(truncate=False)

+--------------+-----------------+-----------+
|database      |tableName        |isTemporary|
+--------------+-----------------+-----------+
|projeto_vendas|bronze_clientes  |false      |
|projeto_vendas|bronze_produtos  |false      |
|projeto_vendas|bronze_vendas    |false      |
|projeto_vendas|bronze_vendedores|false      |
|projeto_vendas|silver_clientes  |false      |
|projeto_vendas|silver_produtos  |false      |
+--------------+-----------------+-----------+



In [0]:
# =====================================
# ETAPA 10 - LIMPEZA DOS VENDEDORES
# =====================================

df_vendedores_bronze = spark.table(
    "projeto_vendas.bronze_vendedores"
)

df_vendedores_silver = (
    df_vendedores_bronze
    .dropDuplicates(["id_vendedor"])
    .filter(
        col("id_vendedor").isNotNull() &
        col("nome_vendedor").isNotNull()
    )
    .withColumn(
        "nome_vendedor",
        initcap(trim(col("nome_vendedor")))
    )
    .withColumn(
        "regiao",
        initcap(trim(col("regiao")))
    )
)

df_vendedores_silver.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.silver_vendedores"
)

display(df_vendedores_silver)

id_vendedor,nome_vendedor,regiao
1,Ana,Sul
2,Pedro,Sudeste
3,Lucas,Centro-oeste
4,Juliana,Nordeste
5,Marcos,Norte


In [0]:
# =====================================
# LIMPEZA DOS REGISTROS DE VENDAS
# =====================================

df_vendas_bronze = spark.table(
    "projeto_vendas.bronze_vendas"
)

df_vendas_silver = (
    df_vendas_bronze
    .dropDuplicates(["id_venda"])
    .filter(
        col("id_venda").isNotNull() &
        col("id_cliente").isNotNull() &
        col("id_produto").isNotNull() &
        col("id_vendedor").isNotNull() &
        (col("valor_venda") > 0)
    )
)

df_vendas_silver.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.silver_vendas"
)

display(df_vendas_silver)

id_venda,id_cliente,id_produto,id_vendedor,valor_venda
1001,1,101,1,3500.0
1002,2,102,2,2200.0
1003,3,103,3,2800.0
1004,4,104,4,1500.0
1005,5,105,5,950.0
1006,1,106,1,150.0
1007,2,107,2,80.0
1008,3,108,3,750.0
1009,4,101,4,3500.0
1010,5,102,5,2200.0


In [0]:
spark.sql("""
SHOW TABLES IN projeto_vendas
""").show(truncate=False)

+--------------+-----------------+-----------+
|database      |tableName        |isTemporary|
+--------------+-----------------+-----------+
|projeto_vendas|bronze_clientes  |false      |
|projeto_vendas|bronze_produtos  |false      |
|projeto_vendas|bronze_vendas    |false      |
|projeto_vendas|bronze_vendedores|false      |
|projeto_vendas|silver_clientes  |false      |
|projeto_vendas|silver_produtos  |false      |
|projeto_vendas|silver_vendas    |false      |
|projeto_vendas|silver_vendedores|false      |
+--------------+-----------------+-----------+



In [0]:
# =====================================
# ETAPA 11 - CAMADA GOLD
# FATURAMENTO POR PRODUTO
# =====================================

from pyspark.sql.functions import sum, count, desc

# Carregar dados Silver
clientes = spark.table("projeto_vendas.silver_clientes")
produtos = spark.table("projeto_vendas.silver_produtos")
vendas = spark.table("projeto_vendas.silver_vendas")

# Integrar vendas e produtos
df_gold_produtos = (
    vendas
    .join(produtos, "id_produto")
    .groupBy("id_produto", "produto")
    .agg(
        count("id_venda").alias("quantidade_vendas"),
        sum("valor_venda").alias("faturamento_total")
    )
    .orderBy(desc("faturamento_total"))
)

# Salvar tabela Gold
df_gold_produtos.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.gold_faturamento_produtos"
)

display(df_gold_produtos)

id_produto,produto,quantidade_vendas,faturamento_total
101,Notebook,2,7000.0
102,Smartphone,2,4400.0
103,Televisao,1,2800.0
104,Tablet,1,1500.0
105,Monitor,1,950.0
108,Impressora,1,750.0
106,Teclado,1,150.0
107,Mouse,1,80.0


In [0]:
# =====================================
# ETAPA 12 - FATURAMENTO POR VENDEDOR
# =====================================

from pyspark.sql.functions import sum, count, desc

vendedores = spark.table(
    "projeto_vendas.silver_vendedores"
)

vendas = spark.table(
    "projeto_vendas.silver_vendas"
)

df_gold_vendedores = (
    vendas
    .join(vendedores, "id_vendedor")
    .groupBy("id_vendedor", "nome_vendedor", "regiao")
    .agg(
        count("id_venda").alias("quantidade_vendas"),
        sum("valor_venda").alias("faturamento_total")
    )
    .orderBy(desc("faturamento_total"))
)

df_gold_vendedores.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.gold_faturamento_vendedores"
)

display(df_gold_vendedores)

id_vendedor,nome_vendedor,regiao,quantidade_vendas,faturamento_total
4,Juliana,Nordeste,2,5000.0
1,Ana,Sul,2,3650.0
3,Lucas,Centro-oeste,2,3550.0
5,Marcos,Norte,2,3150.0
2,Pedro,Sudeste,2,2280.0


In [0]:
# =====================================
# ETAPA 13 - FATURAMENTO POR REGIAO
# =====================================

df_gold_regioes = (
    df_gold_vendedores
    .groupBy("regiao")
    .agg(
        sum("quantidade_vendas").alias("total_vendas"),
        sum("faturamento_total").alias("faturamento")
    )
    .orderBy(desc("faturamento"))
)

df_gold_regioes.write.format("delta").mode("overwrite").saveAsTable(
    "projeto_vendas.gold_faturamento_regioes"
)

display(df_gold_regioes)

regiao,total_vendas,faturamento
Nordeste,2,5000.0
Sul,2,3650.0
Centro-oeste,2,3550.0
Norte,2,3150.0
Sudeste,2,2280.0


In [0]:
spark.sql("""
SHOW TABLES IN projeto_vendas
""").show(truncate=False)

+--------------+---------------------------+-----------+
|database      |tableName                  |isTemporary|
+--------------+---------------------------+-----------+
|projeto_vendas|bronze_clientes            |false      |
|projeto_vendas|bronze_produtos            |false      |
|projeto_vendas|bronze_vendas              |false      |
|projeto_vendas|bronze_vendedores          |false      |
|projeto_vendas|gold_faturamento_produtos  |false      |
|projeto_vendas|gold_faturamento_regioes   |false      |
|projeto_vendas|gold_faturamento_vendedores|false      |
|projeto_vendas|silver_clientes            |false      |
|projeto_vendas|silver_produtos            |false      |
|projeto_vendas|silver_vendas              |false      |
|projeto_vendas|silver_vendedores          |false      |
+--------------+---------------------------+-----------+



In [0]:
%sql
-- =====================================
-- ETAPA 15 - ANALISE SQL
-- =====================================

SELECT
    produto,
    quantidade_vendas,
    faturamento_total
FROM projeto_vendas.gold_faturamento_produtos
ORDER BY faturamento_total DESC;

produto,quantidade_vendas,faturamento_total
Notebook,2,7000.0
Smartphone,2,4400.0
Televisao,1,2800.0
Tablet,1,1500.0
Monitor,1,950.0
Impressora,1,750.0
Teclado,1,150.0
Mouse,1,80.0


In [0]:
%sql
SELECT
    nome_vendedor,
    regiao,
    quantidade_vendas,
    faturamento_total
FROM projeto_vendas.gold_faturamento_vendedores
ORDER BY faturamento_total DESC;

nome_vendedor,regiao,quantidade_vendas,faturamento_total
Juliana,Nordeste,2,5000.0
Ana,Sul,2,3650.0
Lucas,Centro-oeste,2,3550.0
Marcos,Norte,2,3150.0
Pedro,Sudeste,2,2280.0


In [0]:
%sql
SELECT
    regiao,
    total_vendas,
    faturamento
FROM projeto_vendas.gold_faturamento_regioes
ORDER BY faturamento DESC;

regiao,total_vendas,faturamento
Nordeste,2,5000.0
Sul,2,3650.0
Centro-oeste,2,3550.0
Norte,2,3150.0
Sudeste,2,2280.0
